# Change Data Capture & MERGE

**Change Data Capture (CDC)** is the pattern of tracking row-level changes in a source system — inserts, updates, and deletes — and propagating those changes to a downstream table. `MERGE INTO` is the SQL statement that applies those changes atomically.

Together they are the foundation of **incremental Silver layer processing**: instead of rewriting the entire Silver table from Bronze on every run, you apply only what changed.

In this notebook we'll cover:
- CDC record structure and operation types
- `MERGE INTO` syntax — all WHEN clauses
- Applying CDC changes: handling insert, update, and delete in one statement
- Deduplicating CDC feeds before merging
- `WHEN NOT MATCHED BY SOURCE` — removing stale rows
- SCD Type 1 vs Type 2 patterns

## What is CDC?

Source databases (Postgres, MySQL, Oracle) record every change to a transaction log. **CDC tools** (Debezium, AWS DMS, Fivetran) read that log and emit each change as a record with:

- The **operation type** — `I` (insert), `U` (update), `D` (delete)
- The **after-image** — the row's new values (for insert/update)
- The **before-image** — the row's old values (for update/delete, tool-dependent)
- A **sequence number or timestamp** — ordering within the feed

```
Source DB transaction log
   INSERT cust_id=001, name='Alice'         → CDC record: op=I, cust_id=001, name='Alice'
   UPDATE cust_id=001, name='Alice Smith'   → CDC record: op=U, cust_id=001, name='Alice Smith'
   DELETE cust_id=001                       → CDC record: op=D, cust_id=001
```

These records land in a Bronze table (typically via Auto Loader or Kafka). The Silver layer applies them using `MERGE INTO` to keep the Silver table in sync with the source.

## MERGE INTO — Full Syntax

```sql
MERGE INTO target_table  AS t
USING source_table_or_query AS s
ON    t.key = s.key

-- Key exists in BOTH target and source
WHEN MATCHED AND s.op = 'D' THEN
  DELETE

WHEN MATCHED AND s.op = 'U' THEN
  UPDATE SET
    t.name       = s.name,
    t.updated_at = s.updated_at

-- Key in source but NOT in target
WHEN NOT MATCHED [BY TARGET] AND s.op = 'I' THEN
  INSERT (key, name, updated_at)
  VALUES (s.key, s.name, s.updated_at)

-- Key in target but NOT in source (Delta-specific)
WHEN NOT MATCHED BY SOURCE THEN
  DELETE;
```

### Key Rules

- **Multiple `WHEN MATCHED` clauses** are allowed — they are evaluated **in order**; the first match wins
- Each `WHEN MATCHED` clause must be `DELETE` or `UPDATE SET`
- `WHEN NOT MATCHED` inserts rows from source that don't exist in target — can also be written `WHEN NOT MATCHED BY TARGET`
- `WHEN NOT MATCHED BY SOURCE` is Delta-specific — handles rows in target that are absent from source
- `UPDATE SET *` is a shorthand that maps every source column to the matching target column by name
- The entire MERGE is **atomic** — either all changes commit or none do

> **Exam tip:** The order of `WHEN MATCHED` clauses matters. Put the `DELETE` condition before the generic `UPDATE` — otherwise rows with `op='D'` would match the first `WHEN MATCHED` (update) instead of the intended delete.

## Deduplicating CDC Before MERGE

A CDC batch may contain **multiple changes for the same key** — for example, a customer was updated twice and then deleted within the same batch window:

```
seq=1  op=U  cust_id=001  name='Alice Smith'
seq=2  op=U  cust_id=001  name='Alice J. Smith'
seq=3  op=D  cust_id=001
```

If you MERGE all three records, the result depends on which one is processed last — undefined in a non-sequential MERGE. You must **deduplicate to the latest operation per key** before merging:

```sql
-- Keep only the latest CDC record per key
CREATE OR REPLACE TEMP VIEW latest_changes AS
SELECT * EXCEPT(rn) FROM (
  SELECT *,
         ROW_NUMBER() OVER (
           PARTITION BY cust_id
           ORDER BY seq_num DESC    -- latest first
         ) AS rn
  FROM bronze_cdc
  WHERE batch_id = current_batch   -- only this batch
) WHERE rn = 1;
```

Then MERGE from `latest_changes` — each key appears exactly once.

## WHEN NOT MATCHED BY SOURCE

A standard MERGE handles inserts, updates, and deletes that appear explicitly in the CDC feed. But what about rows in the **target that no longer exist in the source** — not because a DELETE event arrived, but because the full source snapshot no longer includes them?

This arises with **full snapshot CDC** — the source periodically sends a complete snapshot of the table rather than individual change events. Rows absent from the new snapshot should be removed from the target.

```sql
MERGE INTO silver.customers AS t
USING latest_snapshot       AS s
ON    t.cust_id = s.cust_id

WHEN MATCHED AND (
    t.name != s.name OR t.email != s.email
) THEN
  UPDATE SET *

WHEN NOT MATCHED BY TARGET THEN
  INSERT *

WHEN NOT MATCHED BY SOURCE THEN  -- in target but not in snapshot
  DELETE;
```

> `WHEN NOT MATCHED BY SOURCE` is a **Delta Lake extension** — it does not exist in standard SQL or non-Delta Spark tables. When combined with `WHEN NOT MATCHED BY TARGET`, a single MERGE can fully synchronize a target table with a source snapshot.

## SCD Type 1 vs Type 2

**Slowly Changing Dimensions (SCD)** describe how to handle historical changes in dimension tables:

| | SCD Type 1 | SCD Type 2 |
|---|---|---|
| **Strategy** | Overwrite — no history kept | Append new version — full history kept |
| **Rows per entity** | Always 1 | 1 per version (multiple over time) |
| **Implementation** | `MERGE` with `UPDATE SET *` | `MERGE` with INSERT for new version + UPDATE to close old version |
| **Use case** | Corrections, typos, non-historical attributes | Slowly changing attributes where history matters |

### SCD Type 1 — Simple MERGE

```sql
MERGE INTO dim.customers AS t
USING cdc_updates         AS s ON t.cust_id = s.cust_id
WHEN MATCHED     THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;
```

### SCD Type 2 — MERGE with History

```sql
MERGE INTO dim.customers AS t
USING cdc_updates         AS s ON t.cust_id = s.cust_id
                                AND t.is_current = true

-- Close the current version
WHEN MATCHED AND s.op != 'D' THEN
  UPDATE SET t.valid_to = s.updated_at, t.is_current = false

-- Delete: just close, don't insert new
WHEN MATCHED AND s.op = 'D' THEN
  UPDATE SET t.valid_to = s.updated_at, t.is_current = false

-- New or changed record: insert as current version
WHEN NOT MATCHED THEN
  INSERT (cust_id, name, email, valid_from, valid_to, is_current)
  VALUES (s.cust_id, s.name, s.email, s.updated_at, NULL, true);
```

> SCD Type 2 is more complex — in practice, Delta Live Tables' `APPLY CHANGES INTO` (covered in notebook 12) handles SCD Type 2 automatically with a single declarative statement.

## Hands-On: Full CDC Pipeline

In [ ]:
spark.sql("CREATE SCHEMA IF NOT EXISTS cdc_demo")
spark.sql("USE SCHEMA cdc_demo")

# Silver target table — the clean, current state
spark.sql("""
  CREATE OR REPLACE TABLE silver_customers (
    cust_id    STRING,
    name       STRING,
    email      STRING,
    tier       STRING,
    updated_at TIMESTAMP
  ) USING DELTA
""")

# Seed with initial data (as if previous batches already ran)
spark.sql("""
  INSERT INTO silver_customers VALUES
    ('C001', 'Alice',   'alice@ex.com',   'gold',   '2024-01-10 09:00:00'),
    ('C002', 'Bob',     'bob@ex.com',     'silver', '2024-01-10 09:00:00'),
    ('C003', 'Carol',   'carol@ex.com',   'bronze', '2024-01-10 09:00:00')
""")

spark.sql("SELECT * FROM silver_customers ORDER BY cust_id").show()

In [ ]:
# Bronze CDC batch — mix of inserts, updates, deletes, and duplicates
spark.sql("""
  CREATE OR REPLACE TABLE bronze_cdc (
    op         STRING,   -- I=insert, U=update, D=delete
    seq_num    LONG,     -- ordering within the feed
    cust_id    STRING,
    name       STRING,
    email      STRING,
    tier       STRING,
    updated_at TIMESTAMP
  ) USING DELTA
""")

spark.sql("""
  INSERT INTO bronze_cdc VALUES
    -- C001: two updates — keep only the latest (seq_num=2)
    ('U', 1, 'C001', 'Alice Smith', 'alice@ex.com',   'gold',     '2024-01-15 10:00:00'),
    ('U', 2, 'C001', 'Alice Smith', 'alice.s@ex.com', 'platinum', '2024-01-15 11:00:00'),
    -- C002: delete
    ('D', 3, 'C002', NULL, NULL, NULL, '2024-01-15 10:30:00'),
    -- C004: new insert
    ('I', 4, 'C004', 'Dave',  'dave@ex.com',  'bronze', '2024-01-15 12:00:00')
""")

spark.sql("SELECT * FROM bronze_cdc ORDER BY seq_num").show()

In [ ]:
# Step 1: deduplicate — keep latest CDC record per key
spark.sql("""
  CREATE OR REPLACE TEMP VIEW latest_cdc AS
  SELECT * EXCEPT(rn) FROM (
    SELECT *,
           ROW_NUMBER() OVER (
             PARTITION BY cust_id
             ORDER BY seq_num DESC
           ) AS rn
    FROM bronze_cdc
  ) WHERE rn = 1
""")

print("Deduplicated CDC (one record per key):")
spark.sql("SELECT * FROM latest_cdc ORDER BY cust_id").show()

In [ ]:
# Step 2: MERGE — apply all changes atomically
spark.sql("""
  MERGE INTO silver_customers AS t
  USING latest_cdc            AS s
  ON    t.cust_id = s.cust_id

  -- DELETE must come before UPDATE (first match wins)
  WHEN MATCHED AND s.op = 'D' THEN
    DELETE

  WHEN MATCHED AND s.op = 'U' THEN
    UPDATE SET
      t.name       = s.name,
      t.email      = s.email,
      t.tier       = s.tier,
      t.updated_at = s.updated_at

  WHEN NOT MATCHED AND s.op = 'I' THEN
    INSERT (cust_id, name, email, tier, updated_at)
    VALUES (s.cust_id, s.name, s.email, s.tier, s.updated_at)
""")

print("Silver after MERGE:")
spark.sql("SELECT * FROM silver_customers ORDER BY cust_id").show()

In [ ]:
# Verify expected results:
# C001 → updated to platinum tier with new email
# C002 → deleted
# C003 → unchanged
# C004 → inserted
result = spark.sql("SELECT * FROM silver_customers ORDER BY cust_id")
result.show()

assert result.count() == 3, "Expected 3 rows (C001, C003, C004)"
assert result.filter("cust_id = 'C002'").count() == 0, "C002 should be deleted"
c001 = result.filter("cust_id = 'C001'").collect()[0]
assert c001['tier'] == 'platinum', "C001 should be platinum"
print("All assertions passed")

In [ ]:
# WHEN NOT MATCHED BY SOURCE — full snapshot sync
# Simulate a full snapshot that excludes C003 (it was deleted upstream)
spark.sql("""
  CREATE OR REPLACE TEMP VIEW full_snapshot AS
  SELECT * FROM VALUES
    ('C001', 'Alice Smith', 'alice.s@ex.com', 'platinum', CURRENT_TIMESTAMP()),
    ('C004', 'Dave',        'dave@ex.com',    'bronze',   CURRENT_TIMESTAMP())
  AS t(cust_id, name, email, tier, updated_at)
""")

spark.sql("""
  MERGE INTO silver_customers AS t
  USING full_snapshot          AS s
  ON    t.cust_id = s.cust_id

  WHEN MATCHED AND (
      t.name != s.name OR t.email != s.email OR t.tier != s.tier
  ) THEN UPDATE SET *

  WHEN NOT MATCHED BY TARGET THEN INSERT *

  WHEN NOT MATCHED BY SOURCE THEN DELETE   -- C003 not in snapshot → remove
""")

print("After full snapshot sync (C003 removed):")
spark.sql("SELECT * FROM silver_customers ORDER BY cust_id").show()

In [ ]:
# MERGE history — operationMetrics shows rows inserted/updated/deleted
spark.sql("""
  DESCRIBE HISTORY silver_customers
""").select(
    "version", "operation", "operationMetrics"
).show(5, truncate=False)

In [ ]:
# Cleanup
spark.sql("DROP SCHEMA IF EXISTS cdc_demo CASCADE")

## MERGE Performance Tips

MERGE is powerful but can be slow on large tables if not optimised:

**1. Filter the source early**
```sql
USING (SELECT * FROM bronze_cdc WHERE batch_id = :current_batch) AS s
```
Never MERGE the entire CDC history — only the current batch.

**2. Use `INSERT_ONLY` hint when there are no updates or deletes**
```sql
MERGE INTO target AS t
USING source      AS s
ON    t.key = s.key
WHEN NOT MATCHED THEN INSERT *;
-- Delta automatically applies the insert-only optimization
```

**3. Cluster or Z-ORDER on the join key**
```sql
OPTIMIZE silver_customers ZORDER BY (cust_id);
-- or use liquid clustering: CLUSTER BY (cust_id)
```
Data skipping becomes effective when the join key is clustered — Delta can skip files that don't contain matching keys.

**4. Enable deletion vectors**
```sql
ALTER TABLE silver_customers
SET TBLPROPERTIES ('delta.enableDeletionVectors' = 'true');
```
With deletion vectors, the UPDATE and DELETE parts of a MERGE write sidecars instead of rewriting full files — significantly faster.

## Summary

- **CDC** captures row-level changes (insert/update/delete) from source systems. Each change record carries an operation type (`I`, `U`, `D`), the new values, and a sequence number for ordering.
- **`MERGE INTO`** applies CDC changes atomically. Clauses: `WHEN MATCHED` (key in both — UPDATE or DELETE), `WHEN NOT MATCHED [BY TARGET]` (key in source only — INSERT), `WHEN NOT MATCHED BY SOURCE` (key in target only — Delta-specific, UPDATE or DELETE).
- Multiple `WHEN MATCHED` clauses are evaluated **in order** — put `DELETE` before `UPDATE`.
- **Always deduplicate** the CDC batch before MERGE using `ROW_NUMBER() OVER (PARTITION BY key ORDER BY seq DESC)`. Multiple changes per key in one batch will produce non-deterministic results without deduplication.
- **`WHEN NOT MATCHED BY SOURCE`** enables full snapshot synchronization — rows in the target that are absent from the source snapshot are deleted.
- **SCD Type 1** — overwrite with MERGE UPDATE (no history). **SCD Type 2** — insert new version + close old version; use Delta Live Tables `APPLY CHANGES INTO` for automatic SCD Type 2 (notebook 12).
- MERGE performance: filter source to current batch, Z-ORDER/cluster on the join key, enable deletion vectors.

Next: `11-structured-streaming-databricks.ipynb` — Structured Streaming patterns specific to Databricks: stateful aggregations, watermarks, and streaming joins.